# V1.0.2-b — Chief Complaint System Classifier

Predicts `chief_complaint_system` (14 body-system classes) from `chief_complaint_raw` free-text only.  
Pipeline: BioBERT → PCA(10) → CatBoost MultiClass — 5-Fold Stratified CV

In [ ]:
# Step 1 — Load & Prepare the 2-Column Dataset
import pandas as pd
import numpy as np
from pathlib import Path

ML = Path("..")

print("Loading chief_complaints.csv...")
cc_df = pd.read_csv(ML / "dataset/raw/chief_complaints.csv")

print(f"Raw shape: {cc_df.shape}")
print(cc_df.head(3))

# Drop patient_id — not a feature
cc_df = cc_df.drop(columns=["patient_id"])

# Verify no nulls
print(f"\nNull counts:\n{cc_df.isnull().sum()}")

# Class distribution
print(f"\nClass distribution ({cc_df['chief_complaint_system'].nunique()} classes):")
print(cc_df["chief_complaint_system"].value_counts())

# Save the clean 2-column parquet
out_path = ML / "dataset/processed/cc_system_dataset.parquet"
cc_df.to_parquet(out_path, index=False)
print(f"\nSaved to: {out_path}")
print(f"Final shape: {cc_df.shape}")

In [ ]:
# Step 2 — BioBERT Embedding Extraction
import torch
import json
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

# Load v1.0.2-b config
with open(ML / "params/v1.0.2-b.json", "r") as f:
    config = json.load(f)

model_name = config["BERT_MODEL_NAME"]   # dmis-lab/biobert-v1.1
batch_size = config["BERT_BATCH_SIZE"]   # 64
max_len    = config["BERT_MAX_LENGTH"]   # 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"BioBERT running on: {device}")

print(f"Loading {model_name}...")
tokenizer  = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name).to(device)
bert_model.eval()

texts = cc_df["chief_complaint_raw"].tolist()
all_embeddings = []

print(f"\nExtracting embeddings for {len(texts)} texts...")
with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        ).to(device)
        outputs = bert_model(**encoded)
        # [CLS] token = index 0 — BioBERT's full-sentence summary
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)

final_embeddings = np.vstack(all_embeddings)
print(f"\nDone. Embedding matrix shape: {final_embeddings.shape}")  # expect (100000, 768)


In [ ]:
# Step 3 — PCA Compression (768 → 10)
import joblib
from sklearn.decomposition import PCA

n_components = config["PCA_COMPONENTS"]  # 10
print(f"Compressing 768-D → {n_components}-D...")

# Fit a NEW, SEPARATE PCA — not the same as pca_v1.0.2.pkl
pca_b = PCA(n_components=n_components, random_state=42)
compressed = pca_b.fit_transform(final_embeddings)

print(f"Compressed shape: {compressed.shape}")
print(f"\nExplained variance per component:\n{pca_b.explained_variance_ratio_}")
print(f"Total variance retained: {pca_b.explained_variance_ratio_.sum() * 100:.1f}%")

# Build the clean PCA DataFrame
pca_cols = [f"biobert_pca_{i+1}" for i in range(n_components)]
pca_df = pd.DataFrame(compressed, columns=pca_cols)

print(f"\nSample PCA output:\n{pca_df.head(3)}")

# Save to v1.0.2-b dedicated folder
os.makedirs(ML / "models/v1.0.2-b", exist_ok=True)
pca_path = ML / "models/v1.0.2-b/pca_v1.0.2-b.pkl"
joblib.dump(pca_b, pca_path)
print(f"\nPCA saved to: {pca_path}")

In [ ]:
# Step 4 — Define X and Y
X = pca_df.copy()
Y = cc_df["chief_complaint_system"].reset_index(drop=True)

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")
print(f"\nClasses ({Y.nunique()}):\n{Y.value_counts()}")


In [ ]:
# Step 5 — 5-Fold Stratified Cross Validation
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier

os.makedirs(ML / "models/v1.0.2-b", exist_ok=True)
os.makedirs(ML / "metrics/v1.0.2-b", exist_ok=True)

cv_folds = config["CV_FOLDS"]
skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=config["RANDOM_SEED"])

# dtype=object because Y contains strings
oof_predictions = np.empty(len(Y), dtype=object)
best_score = 0
best_fold  = 0
last_model = None

print(f"--- STARTING {cv_folds}-FOLD CV (14-class CC System Classifier) ---\n")

for fold, (train_idx, test_idx) in enumerate(skf.split(X, Y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]

    model = CatBoostClassifier(**config["CATBOOST_PARAMS"])
    # no cat_features — all 10 columns are numeric PCA dims
    model.fit(X_train, Y_train, cat_features=[], eval_set=(X_test, Y_test))

    fold_preds = model.predict(X_test).flatten()
    oof_predictions[test_idx] = fold_preds

    score = accuracy_score(Y_test, fold_preds)
    print(f"Fold {fold + 1} Accuracy: {score * 100:.2f}%")

    # Save to v1.0.2-b dedicated folder
    model_path = ML / f"models/v1.0.2-b/model_v1.0.2-b_fold_{fold + 1}.cbm"
    model.save_model(str(model_path))

    if score > best_score:
        best_score = score
        best_fold  = fold + 1

    last_model = model

print("\n" + "=" * 55)
print(f"CHAMPION: Fold {best_fold}  —  {best_score * 100:.2f}%")
print(f"OOF PIPELINE ACCURACY: {accuracy_score(Y, oof_predictions) * 100:.2f}%")
print("=" * 55)

In [ ]:
# Step 6 — Metrics & Visualization
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

metrics_dir = ML / "metrics/v1.0.2-b"

# --- Classification Report ---
report = classification_report(Y, oof_predictions)
print(report)

with open(metrics_dir / "classification_report.txt", "w") as f:
    f.write(f"V1.0.2-b CC SYSTEM CLASSIFIER\n")
    f.write(f"OOF Accuracy: {accuracy_score(Y, oof_predictions) * 100:.2f}%\n\n")
    f.write(report)

# --- Confusion Matrix (14x14) ---
labels = sorted(Y.unique())
cm = confusion_matrix(Y, oof_predictions, labels=labels)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix — CC System Classifier (v1.0.2-b)", fontsize=14, fontweight="bold")
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(metrics_dir / "confusion_matrix.png", bbox_inches="tight", dpi=300)
plt.show()

# --- Feature Importance ---
fi = last_model.get_feature_importance(prettified=True)

plt.figure(figsize=(10, 6))
sns.barplot(x="Importances", y="Feature Id", hue="Feature Id", legend=False,
            data=fi.head(10), palette="viridis")
plt.title("Top 10 PCA Dimensions — CC System Classifier (v1.0.2-b)", fontsize=14, fontweight="bold")
plt.xlabel("Importance Score", fontsize=12)
plt.ylabel("PCA Dimension", fontsize=12)
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig(metrics_dir / "feature_importance.png", bbox_inches="tight", dpi=300)
plt.show()

print("Metrics saved to ml/metrics/v1.0.2-b/")


In [ ]:
# Step 7 — Crown best fold + save full training summary
import shutil
import json

# Copy best fold to production path — all in v1.0.2-b folder
src  = ML / f"models/v1.0.2-b/model_v1.0.2-b_fold_{best_fold}.cbm"
dest = ML / "models/v1.0.2-b/model_v1.0.2-b.cbm"
shutil.copy(str(src), str(dest))

# Recompute per-fold accuracies for the summary
fold_accuracies = {}
for fold, (train_idx, test_idx) in enumerate(
    StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=config["RANDOM_SEED"]).split(X, Y)
):
    fold_accuracies[f"fold_{fold+1}"] = round(
        accuracy_score(Y.iloc[test_idx], oof_predictions[test_idx]), 6
    )

summary = {
    "model_version": "v1.0.2-b",
    "best_fold": best_fold,
    "best_fold_accuracy": round(best_score, 6),
    "oof_pipeline_accuracy": round(accuracy_score(Y, oof_predictions), 6),
    "fold_accuracies": fold_accuracies
}

summary_path = ML / "metrics/v1.0.2-b/training_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print(f"Champion: Fold {best_fold}  —  {best_score * 100:.2f}%")
print(f"OOF Pipeline Accuracy: {accuracy_score(Y, oof_predictions) * 100:.2f}%")
print(f"\nSummary saved to: {summary_path}")
print(f"\nArtifacts ready:")
print(f"  PCA:     ml/models/v1.0.2-b/pca_v1.0.2-b.pkl")
print(f"  Model:   ml/models/v1.0.2-b/model_v1.0.2-b.cbm")
print(f"  Summary: ml/metrics/v1.0.2-b/training_summary.json")

In [ ]:
# Release GPU memory after training — prevents CUDA error 999 in next notebook
import torch
import gc

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

gc.collect()
print(f"GPU memory released.")
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**2:.1f} MB")
